### Adult Income Data Cleaning

In [34]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os


In [35]:
# download the official UCI files
adult_data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
adult_test_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"


df_train = pd.read_csv(adult_data_url, header=None)

df_test = pd.read_csv(adult_test_url, header=None, skiprows=1)

print(df_train.shape)
print(df_test.shape)

(32561, 15)
(16281, 15)


In [36]:
df_train.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [37]:
# Give the columns names

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df_train.columns = columns
df_test.columns = columns

print(df_train.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [38]:
# Add train test information
df_train["source_split"] = "train"
df_test["source_split"] = "test"

print(df_train["source_split"].value_counts())
print(df_test["source_split"].value_counts())

source_split
train    32561
Name: count, dtype: int64
source_split
test    16281
Name: count, dtype: int64


In [39]:
# combine both dataset
df = pd.concat([df_train, df_test], ignore_index=True)

print("Dataset shape:", df.shape)

Dataset shape: (48842, 16)


In [40]:
# check the combine dataset
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source_split
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train


In [41]:
# basic dataset inspection
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 48842
Columns: 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       48842 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education_num   48842 non-null  int64 
 5   marital_status  48842 non-null  object
 6   occupation      48842 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital_gain    48842 non-null  int64 
 11  capital_loss    48842 non-null  int64 
 12  hours_per_week  48842 non-null  int64 
 13  native_country  48842 non-null  object
 14  income          48842 non-null  object
 15  source_split    48842 non-null  object
dtypes: int64(6), object(10)
memory usage: 6.0+ MB


In [42]:
# save raw combined dataset

df.to_csv("raw_adult_dataset.csv", index=False)

In [43]:
# Data Cleaning

# Check UCI missing-value marker '?'

question_mark_counts = (df == "?").sum()

print(question_mark_counts)

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
source_split      0
dtype: int64


In [44]:
# Convert empty or whitespace-only text values to missing values

text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].replace(r"^\s*$", np.nan, regex=True)

print(df.isna().sum())

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
source_split      0
dtype: int64


In [45]:
# Handle missing categorical values

missing_columns = ["workclass", "occupation", "native_country"]

for column in missing_columns:
    df[column] = df[column].fillna("Unknown")

print(df[missing_columns].isna().sum())

workclass         0
occupation        0
native_country    0
dtype: int64


In [46]:
# Convert expected numerical columns

numeric_columns = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(df[numeric_columns].dtypes)

age               int64
fnlwgt            int64
education_num     int64
capital_gain      int64
capital_loss      int64
hours_per_week    int64
dtype: object


In [47]:
# Check missing/invalid numeric values after conversion

print(df[numeric_columns].isna().sum())

age               0
fnlwgt            0
education_num     0
capital_gain      0
capital_loss      0
hours_per_week    0
dtype: int64


In [48]:
# Check impossible core values

print("Age below 0:", (df["age"] < 0).sum())
print("Education number below 0:", (df["education_num"] < 0).sum())
print("Hours per week below 0:", (df["hours_per_week"] < 0).sum())

Age below 0: 0
Education number below 0: 0
Hours per week below 0: 0


In [49]:
# Audit duplicate rows

duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 29


In [50]:
# Normalize income labels

df["income"] = df["income"].str.replace(".", "", regex=False)

print(df["income"].value_counts())

income
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [51]:
# Fix high_income column

df["high_income"] = (
    df["income"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .eq(">50K")
    .astype(int)
)

print(pd.crosstab(df["income"], df["high_income"]))

high_income      0      1
income                   
<=50K        37155      0
>50K             0  11687


In [52]:
print(df.loc[df["income"] == ">50K", ["income", "high_income"]].head(10))

Empty DataFrame
Columns: [income, high_income]
Index: []


In [53]:
# Create age bands

age_bins = [0, 24, 34, 44, 54, 64, 100]

age_labels = [
    "Under 25",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

print(df["age_group"].value_counts().sort_index())

age_group
Under 25     8432
25-34       12577
35-44       12193
45-54        8771
55-64        4782
65+          2087
Name: count, dtype: int64


In [54]:
# Create weekly-hour bands

hour_bins = [0, 20, 34, 40, 49, 59, 100]

hour_labels = [
    "0-20",
    "21-34",
    "35-40",
    "41-49",
    "50-59",
    "60+"
]

df["hours_group"] = pd.cut(
    df["hours_per_week"],
    bins=hour_bins,
    labels=hour_labels,
    include_lowest=True
)

print(df["hours_group"].value_counts().sort_index())

hours_group
0-20      4453
21-34     3942
35-40    26095
41-49     4671
50-59     5828
60+       3853
Name: count, dtype: int64


In [55]:
# Create net capital gain/loss

df["net_capital"] = df["capital_gain"] - df["capital_loss"]

print(df[[
    "capital_gain",
    "capital_loss",
    "net_capital"
]].head())

   capital_gain  capital_loss  net_capital
0          2174             0         2174
1             0             0            0
2             0             0            0
3             0             0            0
4             0             0            0


In [56]:
# Save cleaned dataset

df.to_csv("cleaned_adult_dataset.csv", index=False)



In [57]:

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Shape: (48842, 20)

First 5 rows:
   age          workclass  fnlwgt   education  education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital_status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital_gain  capital_loss  hours_per_week  native_country  income  \
0          2174             0    

In [58]:
check_df = pd.read_csv(
    r"E:\adult_income_eda_project\data\cleaned_adult_dataset.csv"
)

print(check_df["high_income"].value_counts())

high_income
0    48842
Name: count, dtype: int64


In [59]:
# Check the CURRENT dataframe

print("Income values:")
print(df["income"].value_counts())

print("\nHigh income:")
print(df["high_income"].value_counts())

print("\nCross-check:")
print(pd.crosstab(df["income"], df["high_income"]))

Income values:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

High income:
high_income
0    37155
1    11687
Name: count, dtype: int64

Cross-check:
high_income      0      1
income                   
<=50K        37155      0
>50K             0  11687


In [60]:
df.to_csv(
    r"E:\adult_income_eda_project\data\cleaned_adult_dataset.csv",
    index=False
)

print("SAVED:", r"E:\adult_income_eda_project\data\cleaned_adult_dataset.csv")

SAVED: E:\adult_income_eda_project\data\cleaned_adult_dataset.csv


In [61]:
test_df = pd.read_csv(
    r"E:\adult_income_eda_project\data\cleaned_adult_dataset.csv"
)

print(test_df["high_income"].value_counts())

print("\nCross-check:")
print(pd.crosstab(test_df["income"], test_df["high_income"]))

high_income
0    37155
1    11687
Name: count, dtype: int64

Cross-check:
high_income      0      1
income                   
<=50K        37155      0
>50K             0  11687


In [62]:
# overall income proportion
income_proportion = (
    df["high_income"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(income_proportion)

high_income
0    76.07
1    23.93
Name: proportion, dtype: float64


In [63]:
income_summary = pd.DataFrame({
    "Income Group": ["<=50K", ">50K"],
    "Count": [
        (df["high_income"] == 0).sum(),
        (df["high_income"] == 1).sum()
    ],
    "Percentage": [
        round((df["high_income"] == 0).mean() * 100, 2),
        round((df["high_income"] == 1).mean() * 100, 2)
    ]
})

print(income_summary)

  Income Group  Count  Percentage
0        <=50K  37155       76.07
1         >50K  11687       23.93
